In [6]:
import pandas as pd
import numpy as np

# Load the MEPS 2022 Full Year Consolidated file (XLSX version)
df = pd.read_excel(r'C:\Users\anikh\OneDrive\Desktop\preventive-care-gaps\data\raw\h243.xlsx',
                   engine='openpyxl')

print("Shape:", df.shape)
print("\nFirst 3 rows (first 10 columns):")
df.iloc[:3, :10]

Shape: (22431, 1420)

First 3 rows (first 10 columns):


,DUID,PID,DUPERSID,PANEL,DATAYEAR,FAMID31,FAMID42,FAMID53,FAMID22,FAMIDYR
0,2460002,101,2460002101,24,2022,A,A,A,A,A
1,2460006,101,2460006101,24,2022,A,A,A,A,A
2,2460006,102,2460006102,24,2022,A,A,A,A,A


In [7]:
# How many unique people?
print("Unique DUPERSID:", df['DUPERSID'].nunique())

# What does the age distribution look like?
print("\nAge stats:")
print(df['AGE22X'].describe())

# Check for MEPS missing value codes
# MEPS uses negative numbers as codes:
# -1 = Inapplicable
# -7 = Don't know
# -8 = Not ascertained  
# -9 = Not reported
# All of these need to be changed to NaN before doing any analysis can take place
print("\nNegative values in AGE22X:")
print(df[df['AGE22X'] < 0]['AGE22X'].value_counts())

Unique DUPERSID: 22431

Age stats:
count    22431.000000
mean        42.963756
std         23.998943
min         -1.000000
25%         22.000000
50%         44.000000
75%         63.000000
max         85.000000
Name: AGE22X, dtype: float64

Negative values in AGE22X:
AGE22X
-1    195
Name: count, dtype: int64


In [9]:
# Print ALL column names - look for names concerning preventative care
for col in sorted(df.columns):
    print(col)

ACCELI42
ACTDTY31
ACTDTY42
ACTDTY53
ACTLIM31
ADACLS42
ADAGE42
ADASKALC42
ADBLDS42
ADBMI42
ADBNDN42
ADBPCK42
ADBRST42
ADBRTC42
ADCHLC42
ADCLIM42
ADCLNS42
ADCMPM42
ADCMPY42
ADCOLN42
ADCOMPAN42
ADDAYA42
ADDAYEXER42
ADDEBT42
ADDPRS42
ADDSCU42
ADEFRT42
ADEMLS42
ADENGY42
ADFLST42
ADGENH42
ADHDADDX
ADHDAGED
ADHOPE42
ADINTR42
ADISOL42
ADLANG42
ADLATERENT42
ADLATEUTIL42
ADLEFTOUT42
ADLHLP31
ADMDVT42
ADMINSEXER42
ADMISSCCLN42
ADMMGR42
ADMNTRT42
ADMOOD42
ADMWCF42
ADNERV42
ADNOAP42
ADNUMDRK42
ADOFTALC42
ADOFTB42
ADOSTP42
ADPAIN42
ADPAP42
ADPAPG42
ADPCFL42
ADPNEU42
ADPROBTRT42
ADPROS42
ADPROX42
ADPRST42
ADPSAG42
ADQTHP42
ADQTMD42
ADQTTB42
ADREST42
ADRNK442_M20
ADRNK542_M20
ADSAD42
ADSEX42
ADSGMD42
ADSHNG42
ADSLEEP42
ADSOCA42
ADSTAL42
ADTBAC42
ADTRTEXP42
ADUNABTRT42
ADUNEXPEXP42
ADUTRM42
ADWGHD42
ADWKLM42
ADWRTH42
ADWTAD42
AFRDCA42
AFRDDN42
AFRDPM42
AFTHOU42
AGE22X
AGE31X
AGE42X
AGE53X
AGELAST
AIDHLP31
ALIIMP22
ALIMP22X
ANGIAGED
ANGIDX
ANYLMI22
ARTHAGED
ARTHDX
ARTHTYPE
ASACUT31
ASATAK31
ASDALY31
ASE

In [10]:
# Confirm key variables exist and preview their values
key_vars = [
    # Outcome variables — preventive care utilization
    'ADPAP42',      # Pap smear
    'ADMMGR42',     # Mammogram
    'ADCOLN42',     # Colorectal/colonoscopy screening
    'ADBLDS42',     # Blood stool test
    'ADCHLC42',     # Cholesterol checked
    'ADBPCK42',     # Blood pressure checked
    'ADFLST42',     # Flu shot
    
    # Delayed care due to cost
    'DLAYCA42',     # Delayed care due to cost
    
    # Patient Demographics
    'AGE22X',       # Age
    'SEX',          # Sex
    'RACETHX',      # Race/ethnicity
    'HISPANX',      # Hispanic
    
    # Socioeconomic Status
    'POVCAT22',     # Poverty category (income tier)
    'EDUCYR',       # Education years
    
    # Insurance Coverage
    'INSURC22',     # Insurance coverage category
    
    # Patient weight
    'PERWT22F',     # Person weight for analysis
    
    # Geography
    'REGION22',     # Census region
]

# Check which ones exist
missing = [v for v in key_vars if v not in df.columns]
found = [v for v in key_vars if v in df.columns]

print("Found:", found)
print("\nMissing:", missing)

Found: ['ADPAP42', 'ADMMGR42', 'ADCOLN42', 'ADBLDS42', 'ADCHLC42', 'ADBPCK42', 'ADFLST42', 'DLAYCA42', 'AGE22X', 'SEX', 'RACETHX', 'HISPANX', 'POVCAT22', 'EDUCYR', 'INSURC22', 'PERWT22F', 'REGION22']

Missing: []


In [11]:
# Preview value distributions for the key variables

vars_to_check = ['ADPAP42', 'ADMMGR42', 'ADCOLN42', 'ADBLDS42', 
                 'ADCHLC42', 'ADBPCK42', 'ADFLST42', 'DLAYCA42',
                 'SEX', 'RACETHX', 'POVCAT22', 'INSURC22']

for var in vars_to_check:
    print(f"\n--- {var} ---")
    print(df[var].value_counts(dropna=False).sort_index())


--- ADPAP42 ---
ADPAP42
-15      167
-1     17736
 1      2883
 2      1645
Name: count, dtype: int64

--- ADMMGR42 ---
ADMMGR42
-15       57
-1     18978
 1      2511
 2       885
Name: count, dtype: int64

--- ADCOLN42 ---
ADCOLN42
-15      352
-1     15435
 1       128
 2      6516
Name: count, dtype: int64

--- ADBLDS42 ---
ADBLDS42
-15      142
-1     15915
 1      1108
 2       546
 3      4720
Name: count, dtype: int64

--- ADCHLC42 ---
ADCHLC42
-15      263
-1     11132
 1      8758
 2      2278
Name: count, dtype: int64

--- ADBPCK42 ---
ADBPCK42
-15      248
-1     11132
 1      9366
 2      1685
Name: count, dtype: int64

--- ADFLST42 ---
ADFLST42
-15      244
-1     11132
 1      5917
 2      5138
Name: count, dtype: int64

--- DLAYCA42 ---
DLAYCA42
-8       28
-7       24
-1      364
 1     1302
 2    20713
Name: count, dtype: int64

--- SEX ---
SEX
1    10608
2    11823
Name: count, dtype: int64

--- RACETHX ---
RACETHX
1     4883
2    12211
3     3244
4     1220
5      

In [12]:
# Check age range and SEX distribution for ADULTS only
print("Age distribution:")
print(df[df['AGE22X'] >= 18]['AGE22X'].describe())

print("\nSEX for adults 18+:")
adults = df[df['AGE22X'] >= 18]
print(adults['SEX'].value_counts())
print("\nTotal adults:", len(adults))

# Check how many adults were asked each screening question
# (i.e., NOT inapplicable/-1)
print("\nAdults asked each screening (not -1 or -15):")
screening_vars = ['ADPAP42', 'ADMMGR42', 'ADCOLN42', 'ADBLDS42', 
                  'ADCHLC42', 'ADBPCK42', 'ADFLST42']
for var in screening_vars:
    asked = adults[(adults[var] != -1) & (adults[var] != -15)]
    print(f"  {var}: {len(asked):,} asked")

Age distribution:
count    17909.000000
mean        51.606232
std         18.539015
min         18.000000
25%         36.000000
50%         53.000000
75%         67.000000
max         85.000000
Name: AGE22X, dtype: float64

SEX for adults 18+:
SEX
2    9654
1    8255
Name: count, dtype: int64

Total adults: 17909

Adults asked each screening (not -1 or -15):
  ADPAP42: 4,528 asked
  ADMMGR42: 3,396 asked
  ADCOLN42: 6,644 asked
  ADBLDS42: 6,374 asked
  ADCHLC42: 11,036 asked
  ADBPCK42: 11,051 asked
  ADFLST42: 11,055 asked
